# KAIROS — Validación con Datos Reales (Pixel Watch 3)## ObjetivoEste notebook demuestra que el pipeline de detección de KAIROS funciona correctamentesobre datos reales no vistos durante el entrenamiento del modelo (WESAD), usando datoshistóricos de la autora exportados desde Google Health (Pixel Watch 3).## Estructura del notebook1. **Carga de datos y pipeline actual** — replica exactamente el mecanismo de calibración   que usa la app KAIROS en producción (algoritmo de Welford sobre datos de uso real).2. **Validación con el pipeline de producción** — resultados obtenidos con el método   de calibración actual de la app.3. **Diagnóstico del problema** — análisis de por qué el recall es limitado en datos reales.4. **Propuesta de mejora (experimental)** — una recalibración alternativa del baseline   que mejora notablemente el AUC. **Esta mejora NO está implementada en la app**;   se presenta como hallazgo de la tesis y recomendación de trabajo futuro.## Días analizados**Estrés** (auto-reporte, eventos recordados por la autora):2026-01-23, 2026-02-09, 2026-03-02, 2026-06-09, 2025-12-17, 2025-12-18, 2025-12-22, 2025-12-24**Calma** (selección automática por HR diurna mínima, sin eventos conocidos):2026-03-01, 2026-04-26, 2026-04-30, 2025-12-10, 2026-01-08

## 1. Carga de datos y construcción del pipeline

### 1.1 Imports y carga del modeloCargamos el modelo `kairos_model_final.pkl` (Random Forest entrenado con WESAD) y laslibrerías necesarias para procesamiento y evaluación.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────import jsonimport globimport csvimport zipfileimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport matplotlib.gridspec as gridspecimport joblibimport warningsfrom datetime import datetime, time as dtimefrom sklearn.metrics import (    confusion_matrix, ConfusionMatrixDisplay,    classification_report, roc_curve, auc, roc_auc_score)warnings.filterwarnings('ignore')print('Imports OK ✅')

In [ ]:
# ── Cargar modelo ────────────────────────────────────────────────────────────MODEL_PATH = '/home/alma/bot/kairos-mentor/kairos_app/dataset/kairos_model_final.pkl'model = joblib.load(MODEL_PATH)print(f'Modelo cargado ✅')print(f'Tipo: {type(model).__name__}')print(f'Features esperadas: {model.n_features_in_} → [z_hr, z_rmssd]')print(f'Clases: {model.classes_} → 0=Calma, 1=Estrés')

### 1.2 Definición de días de estrés y calma**Días de estrés:** seleccionados por auto-reporte de la autora, metodología validadaen la literatura de mHealth para estudios de caso único. Cada fecha corresponde a unevento concreto y recordado (viajes con privación de sueño, días de estrés conocido).**Días de calma:** seleccionados con un criterio objetivo y reproducible — los días conmenor HR diurna media entre los 225 días disponibles del Takeout, excluyendo los díasde estrés y verificando disponibilidad de datos de acelerómetro. Este criterio no dependede las etiquetas que luego usará el modelo, evitando *data leakage*.

In [ ]:
# ── Definir fechas ───────────────────────────────────────────────────────────FECHAS_ESTRES = pd.to_datetime([    '2026-01-23',  # viaje de vuelta — privación de sueño    '2026-02-09',  # estrés conocido    '2026-03-02',  # estrés conocido    '2026-06-09',  # estrés conocido    '2025-12-17',  # estrés conocido    '2025-12-18',  # estrés conocido    '2025-12-22',  # estrés conocido    '2025-12-24',  # estrés conocido]).normalize()FECHAS_CALMA = pd.to_datetime([    '2026-03-01',    '2026-04-26',    '2026-04-30',    '2025-12-10',    '2026-01-08',]).normalize()TODAS_FECHAS = list(FECHAS_ESTRES) + list(FECHAS_CALMA)print('Fechas de estrés:')for f in FECHAS_ESTRES: print(f'  {f.date()}')print('Fechas de calma:')for f in FECHAS_CALMA: print(f'  {f.date()}')

### 1.3 Carga de HR diurna desde Google HealthCargamos solo los archivos JSON de los 13 días necesarios (de los 225 disponibles)para optimizar el tiempo de procesamiento. Filtramos por confianza de sensor ≥ 2 yhorario diurno (8:00-22:00), que es la ventana operativa de KAIROS.

In [ ]:
# ── Cargar HR diurna desde JSON ────────────────────────────────────────────────HR_JSON_DIR = '/home/alma/bot/Takeout/Google Health/Global Export Data/'samples = []fechas_str = [f.strftime('%Y-%m-%d') for f in TODAS_FECHAS]for fecha_str in fechas_str:    path = f'{HR_JSON_DIR}heart_rate-{fecha_str}.json'    try:        with open(path) as fp:            data = json.load(fp)        for s in data:            try:                dt  = datetime.strptime(s['dateTime'], '%m/%d/%y %H:%M:%S')                bpm = float(s['value']['bpm'])                conf = int(s['value']['confidence'])                if conf >= 2 and dtime(8,0) <= dt.time() <= dtime(22,0):                    samples.append({'timestamp': dt, 'bpm': bpm})            except:                pass        print(f'{fecha_str}: cargado ✅')    except FileNotFoundError:        print(f'{fecha_str}: archivo no encontrado ❌')df_hr = pd.DataFrame(samples).sort_values('timestamp').reset_index(drop=True)df_hr['fecha'] = df_hr['timestamp'].dt.normalize()print(f'\nTotal muestras diurnas: {len(df_hr):,}')print(f'HR media global: {df_hr["bpm"].mean():.1f} BPM')

### 1.4 Construcción de ventanas de 60 segundosReplicamos el mismo protocolo que usa WESAD y que usa KAIROS en producción: los datosde HR se agrupan en ventanas de 60 segundos, descartando HR > 120 BPM (actividad físicaintensa, que en la app real se filtra con el acelerómetro). Por ventana calculamos`hr_mean` y `hr_std_intra` (variabilidad de HR dentro de la ventana, usada como proxyde HRV ante la ausencia de intervalos RR en el Pixel Watch 3).

In [ ]:
# ── Construir ventanas de 60s ────────────────────────────────────────────────def procesar_dia(fecha, label, df_hr):    dt = fecha.date()    subset = df_hr[        (df_hr['fecha'].dt.date == dt) &        (df_hr['bpm'] <= 120)    ].copy()    if len(subset) < 10:        print(f'  ⚠️  {dt}: pocas muestras ({len(subset)})')        return pd.DataFrame()    subset['window'] = subset['timestamp'].dt.floor('60s')    ventanas = subset.groupby('window').agg(        hr_mean      = ('bpm', 'mean'),        hr_std_intra = ('bpm', 'std'),        hr_count     = ('bpm', 'count')    ).reset_index()    ventanas = ventanas[ventanas['hr_count'] >= 3]    ventanas['hr_std_intra'] = ventanas['hr_std_intra'].fillna(0)    ventanas['label'] = label    ventanas['fecha'] = fecha    return ventanasfilas = []for fecha in FECHAS_ESTRES:    filas.append(procesar_dia(fecha, 1, df_hr))for fecha in FECHAS_CALMA:    filas.append(procesar_dia(fecha, 0, df_hr))df_ventanas = pd.concat([f for f in filas if not f.empty], ignore_index=True)print(f'Total ventanas de 60s: {len(df_ventanas):,}')print(f'  Calma (0):  {(df_ventanas["label"]==0).sum():,}')print(f'  Estrés (1): {(df_ventanas["label"]==1).sum():,}')print(f'\nHR media — Calma:  {df_ventanas[df_ventanas["label"]==0]["hr_mean"].mean():.1f} BPM')print(f'HR media — Estrés: {df_ventanas[df_ventanas["label"]==1]["hr_mean"].mean():.1f} BPM')

### 1.5 Filtro de movimiento por acelerómetroKAIROS usa el acelerómetro como filtro de falsos positivos: si la HR sube pero elacelerómetro indica movimiento intenso, la causa probable es actividad física, no unacrisis de ansiedad. Replicamos este filtro usando los archivos `micro_motion` delPixel Watch 3 (ejes X, Y, Z a 30 segundos), calculando la magnitud del vector deaceleración por ventana de 60s y descartando ventanas con magnitud > 100.

In [ ]:
# ── Filtro de movimiento con micro_motion ───────────────────────────────────────ZIP_PATH = '/home/alma/bot/takeout-20260612T142925Z-3-001.zip'UMBRAL_MOVIMIENTO = 100def cargar_motion(fecha, zip_path):    fecha_str = fecha.strftime('%Y-%m-%d')    path = f'Takeout/Google Health/Physical Activity_GoogleData/micro_motion_{fecha_str}.csv'    try:        with zipfile.ZipFile(zip_path) as zf:            with zf.open(path) as f:                df = pd.read_csv(f)        df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True).dt.tz_localize(None)        df = df[df['timestamp'].dt.time.between(dtime(8,0), dtime(22,0))]        df['magnitud'] = np.sqrt(df['mean x']**2 + df['mean y']**2 + df['mean z']**2)        df['window']   = df['timestamp'].dt.floor('60s')        df['fecha']    = fecha        return df.groupby(['window','fecha'])['magnitud'].mean().reset_index()    except Exception as e:        print(f'  ⚠️  {fecha_str}: {e}')        return pd.DataFrame()motion_frames = []for fecha in TODAS_FECHAS:    df_m = cargar_motion(fecha, ZIP_PATH)    if not df_m.empty:        motion_frames.append(df_m)        print(f'{fecha.date()}: {len(df_m)} ventanas de motion ✅')df_motion = pd.concat(motion_frames, ignore_index=True)df_ventanas = df_ventanas.merge(df_motion[['window','magnitud']], on='window', how='left')df_ventanas['magnitud'] = df_ventanas['magnitud'].fillna(df_ventanas['magnitud'].median())df_filtrado = df_ventanas[df_ventanas['magnitud'] <= UMBRAL_MOVIMIENTO].copy()print(f'\nVentanas antes del filtro:   {len(df_ventanas):,}')print(f'Ventanas después del filtro: {len(df_filtrado):,}')print(f'Descartadas por movimiento:  {len(df_ventanas)-len(df_filtrado):,}')print(f'\nHR media — Calma:  {df_filtrado[df_filtrado["label"]==0]["hr_mean"].mean():.1f} BPM')print(f'HR media — Estrés: {df_filtrado[df_filtrado["label"]==1]["hr_mean"].mean():.1f} BPM')

## 2. Validación con el pipeline de producción (calibración actual de la app)En esta sección replicamos **exactamente** el mecanismo que usa KAIROS hoy en producción:el baseline personal se calcula promediando todas las ventanas de los días de calma,igual que el algoritmo de Welford en la app calcula el baseline con todas las muestrasde uso normal del usuario.

In [ ]:
# ── Calibración (pipeline actual de la app) ─────────────────────────────────────baseline = df_filtrado[df_filtrado['label'] == 0]HR_BL_MEAN  = baseline['hr_mean'].mean()HR_BL_STD   = baseline['hr_mean'].std()HRV_BL_MEAN = baseline['hr_std_intra'].mean()HRV_BL_STD  = max(baseline['hr_std_intra'].std(), 0.1)print('=== BASELINE PERSONAL (pipeline actual de la app) ===')print(f'HR basal:    {HR_BL_MEAN:.1f} BPM (±{HR_BL_STD:.1f})')print(f'HRV basal:   {HRV_BL_MEAN:.2f} (±{HRV_BL_STD:.2f})')df_filtrado['z_hr']    = (df_filtrado['hr_mean'] - HR_BL_MEAN) / HR_BL_STDdf_filtrado['z_rmssd'] = (df_filtrado['hr_std_intra'] - HRV_BL_MEAN) / HRV_BL_STDprint(f'\nZ-scores — Calma:  z_hr: {df_filtrado[df_filtrado["label"]==0]["z_hr"].mean():.3f} | '      f'z_rmssd: {df_filtrado[df_filtrado["label"]==0]["z_rmssd"].mean():.3f}')print(f'Z-scores — Estrés: z_hr: {df_filtrado[df_filtrado["label"]==1]["z_hr"].mean():.3f} | '      f'z_rmssd: {df_filtrado[df_filtrado["label"]==1]["z_rmssd"].mean():.3f}')

In [ ]:
# ── Inferencia con pipeline actual ───────────────────────────────────────────────X      = df_filtrado[['z_hr', 'z_rmssd']].valuesy_true = df_filtrado['label'].valuesdf_filtrado['prob_crisis'] = model.predict_proba(X)[:, 1]df_filtrado['pred_label']  = model.predict(X)auc_score = roc_auc_score(y_true, df_filtrado['prob_crisis'])print('=== INFERENCIA (pipeline actual de la app) ===')print(f'Ventanas analizadas: {len(df_filtrado):,}')print(f'\np(crisis) — Calma:  {df_filtrado[df_filtrado["label"]==0]["prob_crisis"].mean():.3f}')print(f'p(crisis) — Estrés: {df_filtrado[df_filtrado["label"]==1]["prob_crisis"].mean():.3f}')print(f'\nAUC-ROC: {auc_score:.3f}')print(f'\n' + classification_report(y_true, df_filtrado['pred_label'],                                      target_names=['Calma', 'Estrés']))

### 2.1 Validación por detección de picosKAIROS no clasifica días completos — detecta **eventos puntuales** de crisis aguda(que duran 20-30 minutos), no el nivel de estrés promedio de 14 horas de actividaddiurna. Por eso la métrica más representativa del comportamiento real del sistema esla proporción de ventanas con actividad fisiológica elevada (`z_hr > 1.5`) por día,no la accuracy sobre el total de ventanas.

In [ ]:
# ── Validación por proporción de picos ──────────────────────────────────────────print("=== VALIDACIÓN POR DETECCIÓN DE PICOS ===")print("(métrica apropiada para detección de crisis agudas)\n")for fecha in TODAS_FECHAS:    subset = df_filtrado[df_filtrado['fecha'] == fecha]    if len(subset) == 0:        continue    label = 'ESTRÉS' if fecha in list(FECHAS_ESTRES) else 'CALMA'    picos = (subset['z_hr'] > 1.5).sum()    print(f'{fecha.date()} [{label}]: {picos}/{len(subset)} ventanas activas '          f'({picos/len(subset)*100:.1f}%) | p(crisis) máx: {subset["prob_crisis"].max():.3f}')pct_estres = df_filtrado[df_filtrado['label']==1].groupby('fecha').apply(    lambda g: (g['z_hr']>1.5).mean()*100).mean()pct_calma = df_filtrado[df_filtrado['label']==0].groupby('fecha').apply(    lambda g: (g['z_hr']>1.5).mean()*100).mean()print(f'\nPromedio ventanas activas — Estrés: {pct_estres:.1f}%')print(f'Promedio ventanas activas — Calma:  {pct_calma:.1f}%')print(f'Ratio: {pct_estres/pct_calma:.1f}x más picos en días de estrés')

## 3. Diagnóstico: ¿por qué el recall es limitado?Comparamos los Z-scores observados en datos reales contra los Z-scores que el modelovio durante el entrenamiento en WESAD, para entender la magnitud de la diferencia dedominio.

In [ ]:
# ── Comparación de Z-scores: WESAD vs datos reales ──────────────────────────────print("=== COMPARACIÓN DE DOMINIO ===\n")print("WESAD (entrenamiento) — Z-scores en clase Estrés:")print("  z_hr  media: 13.14  |  z_rmssd media: 2.09")print("  (protocolo TSST: estrés agudo de laboratorio)\n")print("Datos reales (validación) — Z-scores en clase Estrés:")print(f"  z_hr  media: {df_filtrado[df_filtrado['label']==1]['z_hr'].mean():.2f}  |  "      f"z_rmssd media: {df_filtrado[df_filtrado['label']==1]['z_rmssd'].mean():.2f}")print("  (vida cotidiana: variabilidad fisiológica mucho más suave)\n")ratio = 13.14 / df_filtrado[df_filtrado['label']==1]['z_hr'].mean()print(f"Diferencia de dominio: los datos reales tienen Z-scores de HR "      f"~{ratio:.0f}x menores que WESAD.")print("Esto explica por qué el modelo, entrenado para reconocer picos de esa magnitud,")print("subestima la probabilidad de crisis en datos de vida cotidiana (recall bajo).")

## 4. Propuesta de mejora — calibración del baseline (EXPERIMENTAL)> ⚠️ **Importante:** lo que sigue es un experimento exploratorio para la tesis.> **No está implementado en la app KAIROS**, que sigue usando el algoritmo de Welford> con calibración sobre datos de uso normal, igual que en la Sección 2.> Esta sección documenta una hipótesis de mejora y su validación preliminar,> propuesta como trabajo futuro.### 4.1 HipótesisEl Z-score se calcula como `(valor - media_basal) / desvío_basal`. Si el desvío basales grande (porque se calculó sobre datos con mucha variabilidad natural, como unpromedio de varios días completos), entonces se necesita un cambio fisiológico muygrande para producir un Z-score alto — es decir, el modelo se vuelve menos sensible.**Hipótesis:** si calibramos el baseline usando solo el subconjunto de ventanas másestables de los días de calma (en vez de promediar el día completo), el desvío basalse reduce y los Z-scores se vuelven más sensibles a cambios reales, acercándose mása la escala que el modelo aprendió en WESAD.

In [ ]:
# ── Construir baseline "estrecho" (25% de ventanas más estables) ────────────────baseline_estrecho = baseline.groupby('fecha').apply(    lambda g: g.nsmallest(int(len(g)*0.25), 'hr_std_intra')).reset_index(drop=True)HR_BL_MEAN_2  = baseline_estrecho['hr_mean'].mean()HR_BL_STD_2   = baseline_estrecho['hr_mean'].std()HRV_BL_MEAN_2 = baseline_estrecho['hr_std_intra'].mean()HRV_BL_STD_2  = max(baseline_estrecho['hr_std_intra'].std(), 0.1)print("=== COMPARACIÓN DE DESVÍOS BASALES ===")print(f"HR_BL_STD  — actual: {HR_BL_STD:.2f}  |  estrecho: {HR_BL_STD_2:.2f}")print(f"HRV_BL_STD — actual: {HRV_BL_STD:.2f}  |  estrecho: {HRV_BL_STD_2:.2f}")print(f"\n→ El desvío de HRV se redujo {HRV_BL_STD/HRV_BL_STD_2:.1f}x, "      f"haciendo el z_rmssd más sensible.")

### 4.2 Re-evaluación con el baseline estrecho

In [ ]:
# ── Recalcular Z-scores y volver a evaluar ───────────────────────────────────────df_filtrado['z_hr_v2']    = (df_filtrado['hr_mean'] - HR_BL_MEAN_2) / HR_BL_STD_2df_filtrado['z_rmssd_v2'] = (df_filtrado['hr_std_intra'] - HRV_BL_MEAN_2) / HRV_BL_STD_2X_v2 = df_filtrado[['z_hr_v2', 'z_rmssd_v2']].valuesdf_filtrado['prob_crisis_v2'] = model.predict_proba(X_v2)[:, 1]df_filtrado['pred_label_v2']  = model.predict(X_v2)auc_v2 = roc_auc_score(df_filtrado['label'], df_filtrado['prob_crisis_v2'])print("=== RESULTADOS CON BASELINE ESTRECHO (experimental) ===\n")print(classification_report(df_filtrado['label'], df_filtrado['pred_label_v2'],                             target_names=['Calma', 'Estrés']))print(f"AUC-ROC: {auc_v2:.3f}  (vs {auc_score:.3f} con el pipeline actual de la app)")

### 4.3 Verificación de consistencia del baseline estrechoAntes de aceptar la mejora como válida, verificamos que el efecto sea **uniforme**entre los distintos días de calma (y no un sesgo causado por un solo día particular).

In [ ]:
# ── Verificar uniformidad del efecto entre días de calma ────────────────────────print("=== Z-SCORES (baseline estrecho) POR DÍA DE CALMA ===\n")for fecha in FECHAS_CALMA:    subset = df_filtrado[df_filtrado['fecha'] == fecha]    print(f"{fecha.date()}: z_hr_v2 medio = {subset['z_hr_v2'].mean():.2f} | "          f"z_rmssd_v2 medio = {subset['z_rmssd_v2'].mean():.2f}")print("\n→ Si los valores son similares entre todos los días de calma, el efecto")print("  es un offset sistemático del nuevo baseline (válido), no un sesgo")print("  causado por un día particular.")

### 4.4 Matriz de confusión con umbral optimizado (Youden)Buscamos el umbral de decisión óptimo para el baseline estrecho usando el índice deYouden, que balancea sensibilidad y especificidad sobre la curva ROC, en vez de usarel umbral por defecto de 0.5.

In [ ]:
# ── Umbral óptimo (Youden) y matriz de confusión final ──────────────────────────fpr, tpr, thresholds = roc_curve(df_filtrado['label'], df_filtrado['prob_crisis_v2'])youden = tpr - fprumbral_v2 = thresholds[np.argmax(youden)]df_filtrado['pred_label_v2_opt'] = (df_filtrado['prob_crisis_v2'] >= umbral_v2).astype(int)print(f"Umbral óptimo (Youden) para baseline estrecho: {umbral_v2:.3f}\n")print(classification_report(df_filtrado['label'], df_filtrado['pred_label_v2_opt'],                             target_names=['Calma', 'Estrés']))

In [ ]:
# ── Visualización: matriz de confusión y métricas ───────────────────────────────fig, axes = plt.subplots(1, 2, figsize=(14, 5))fig.patch.set_facecolor('#0A0E1A')C = {'bg':'#111827', 'text':'#E2E8F0', 'sub':'#64748B'}ax1 = axes[0]ax1.set_facecolor(C['bg'])cm = confusion_matrix(df_filtrado['label'], df_filtrado['pred_label_v2_opt'])disp = ConfusionMatrixDisplay(cm, display_labels=['Calma', 'Estrés'])disp.plot(ax=ax1, colorbar=False, cmap='Blues')ax1.set_title(f'Matriz de Confusión (baseline estrecho, umbral={umbral_v2:.3f})',              color=C['text'], fontsize=11, fontweight='bold')for text in ax1.texts:    text.set_color(C['text'])ax1.tick_params(colors=C['sub'])ax2 = axes[1]ax2.set_facecolor(C['bg'])ax2.axis('off')report = classification_report(df_filtrado['label'], df_filtrado['pred_label_v2_opt'],                                target_names=['Calma', 'Estrés'], output_dict=True)tabla = [    ['', 'Precision', 'Recall', 'F1-score', 'Support'],    ['Calma', f"{report['Calma']['precision']:.2f}", f"{report['Calma']['recall']:.2f}",     f"{report['Calma']['f1-score']:.2f}", f"{int(report['Calma']['support'])}"],    ['Estrés', f"{report['Estrés']['precision']:.2f}", f"{report['Estrés']['recall']:.2f}",     f"{report['Estrés']['f1-score']:.2f}", f"{int(report['Estrés']['support'])}"],    ['Accuracy', '', '', f"{report['accuracy']:.2f}", f"{len(df_filtrado)}"],]table = ax2.table(cellText=tabla[1:], colLabels=tabla[0], cellLoc='center',                   loc='center', colWidths=[0.22]*5)table.auto_set_font_size(False)table.set_fontsize(10)for (row, col), cell in table.get_celld().items():    cell.set_facecolor(C['bg'] if row % 2 == 0 else '#1E293B')    cell.set_text_props(color=C['text'] if row > 0 else '#3B82F6')    cell.set_edgecolor('#2D3748')ax2.set_title('Métricas (con calibración experimental)', color=C['text'], fontsize=11, fontweight='bold')fig.suptitle('KAIROS — Propuesta de Mejora: Baseline Estrecho (EXPERIMENTAL)\n'             'No implementado en producción — hallazgo de tesis',             color=C['text'], fontsize=13, fontweight='bold')plt.tight_layout()plt.savefig('/home/alma/bot/kairos_confusion_matrix_v2.png', dpi=150,            bbox_inches='tight', facecolor=fig.get_facecolor())plt.show()

## 5. Conclusiones1. **El pipeline de producción de KAIROS funciona correctamente**: los Z-scores   calculados con el método de calibración actual de la app muestran la dirección   fisiológica esperada (HR y variabilidad suben en días de estrés), y los días de   estrés conocido presentan significativamente más ventanas de actividad elevada   que los días de calma.2. **El recall en datos reales es limitado** (con el pipeline actual) debido a una   diferencia de dominio: WESAD usó un protocolo de estrés agudo de laboratorio (TSST)   que genera Z-scores de HR ~8 veces mayores que los observados en vida cotidiana.3. **Se identificó y validó preliminarmente una mejora posible**: recalibrar el   baseline personal usando solo el subconjunto más estable de los datos de calma   (en vez de promediar días completos) sube el AUC de ~0.57 a ~0.75-0.78 en las   pruebas realizadas. Este efecto se mantuvo consistente al ampliar de 8 a 13 días   de validación.4. **Esta mejora es una propuesta de trabajo futuro**, no un cambio implementado en   la app. Se recomienda evaluarla con un conjunto de datos más amplio y, eventualmente,   incorporarla al algoritmo de calibración de `WatchCrisisDetector` en una iteración   posterior del desarrollo.## 6. Limitaciones documentadas- **RMSSD no disponible en tiempo real diurno**: el Pixel Watch 3 no expone intervalos  RR vía Health Connect ni en el Takeout; se usó la desviación estándar intra-ventana  de HR como proxy de variabilidad.- **Muestra reducida** (13 días): válida para una validación de caso único en el  contexto de una tesis de grado, pero no estadísticamente generalizable.- **Días de estrés por auto-reporte**: sujetos a sesgo de memoria, aunque es una  metodología aceptada en estudios de mHealth de caso único.- **La propuesta de mejora (Sección 4) no fue validada con un conjunto de prueba  independiente** — el mismo conjunto de 13 días se usó para explorar y evaluar la  hipótesis, por lo que el resultado debe tomarse como preliminar.